טעינת הנתונים וניקוי
העמודות שאין להם את הדירוג הם נמחקו מהדאטה כי לא תורמות . 
העמודות שיש שיש להם ערכים חסרים בחלק מהמקומות זה הושלם ללא ידוע 
בעמודה של השפות- הושלמו ערכים רקים ללא ידוע , נוצרו עמודות בינאריות לשפות הכי נפוצות (כדי לא ללמד את המודל יותר מדי אופציות בקטגוריה )
בקטגוריה של השנים חילקתי לעשורים והשארתי גם את השנה המדויקת (כי לחלק מהמודלים תלוי אם זה מודלים לינארים או לא ,עדיף ללמוד משנה מדוייקת או מעשורים(תלוי בסוג מודל) ) 


In [6]:
# ==========================================
# חלק 1: ייבוא ספריות, פונקציית העיבוד וקריאת נתונים
# ==========================================
import pandas as pd
import numpy as np
import re

def prepare_data(df):
    """
    פונקציה דטרמיניסטית לעיבוד וניקוי נתונים (השם תוקן לדרישות המרצה).
    כוללת המרת מטבעות חכמה (USD), חוק ה-1000, ועיבוד טקסטים.
    """
    df_clean = df.copy()

    # 1. טיפול בעשור
    if 'startYear' in df_clean.columns:
        df_clean['startYear'] = pd.to_numeric(df_clean['startYear'], errors='coerce')
        decades = [1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
        for dec in decades:
            df_clean[f'is_decade_{dec}'] = ((df_clean['startYear'] // 10) * 10 == dec).astype(int)
        df_clean['is_decade_unknown'] = df_clean['startYear'].isna().astype(int)

    # 2. טיפול בתקציב - כולל המרת מטבעות ל-USD!
    def parse_financial_value(val):
        if pd.isna(val) or str(val).strip() == '' or str(val).lower() == 'unknown':
            return np.nan
            
        val_str = str(val).lower()
        
        # זיהוי מטבע ושער חליפין
        exchange_rate = 1.0  
        if '₹' in val_str or 'inr' in val_str or 'rs' in val_str:
            exchange_rate = 0.012  
        elif '£' in val_str or 'gbp' in val_str:
            exchange_rate = 1.25   
        elif '€' in val_str or 'eur' in val_str:
            exchange_rate = 1.10   
        elif '¥' in val_str or 'jpy' in val_str:
            exchange_rate = 0.0067 

        val_str = re.sub(r'[$,£€¥₹]', '', val_str).replace(',', '')
        
        multiplier, has_explicit = 1, False
        if 'billion' in val_str or 'b' in val_str:
            multiplier, has_explicit = 1000000000, True
        elif 'million' in val_str or 'm' in val_str:
            multiplier, has_explicit = 1000000, True
        elif 'crore' in val_str: 
            multiplier, has_explicit = 10000000, True
        elif 'lakh' in val_str or 'lac' in val_str: 
            multiplier, has_explicit = 100000, True
        elif 'thousand' in val_str or 'k' in val_str:
            multiplier, has_explicit = 1000, True
            
        match = re.search(r'\d+(\.\d+)?([eE][+-]?\d+)?', val_str)
        if match:
            num_str = match.group()
            if 'e-' in num_str:
                num_str = num_str.split('e-')[0]
                multiplier, has_explicit = 1000000, True
                
            result = float(num_str) * multiplier
            if result == 0: return np.nan
            
            if not has_explicit:
                if 0 < result <= 100: result *= 1000000  
                elif 100 < result <= 1000: return np.nan 
                
            return result * exchange_rate
            
        return np.nan

    if 'budget' in df_clean.columns:
        df_clean['budget_parsed'] = df_clean['budget'].apply(parse_financial_value)
        df_clean['is_budget_missing'] = df_clean['budget_parsed'].isna().astype(int)
        df_clean['budget_log'] = np.log1p(df_clean['budget_parsed'])

    # 3. שפות
    if 'Language' in df_clean.columns:
        top_langs = ['English', 'French', 'Hindi', 'Spanish', 'Italian', 'Japanese', 'Tamil', 'German', 'Telugu', 'Malayalam']
        def cat_lang(val):
            res = {f'is_{l}': 0 for l in top_langs}
            res.update({'non_primary_language': 0, 'unknown_language': 0})
            val = str(val).strip()
            if val in ['Not Found', 'unknown', 'nan'] or len(val) > 80:
                res['unknown_language'] = 1
                return pd.Series(res)
            text = val.lower()
            for l in top_langs:
                if l.lower() in text:
                    res[f'is_{l}'] = 1
                    text = text.replace(l.lower(), '')
            if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['non_primary_language'] = 1
            if sum(res.values()) == 0: res['unknown_language'] = 1
            return pd.Series(res)
        df_clean = pd.concat([df_clean, df_clean['Language'].apply(cat_lang)], axis=1)

    # 4. מדינות
    if 'Country' in df_clean.columns:
        top_countries = ['United States', 'United Kingdom', 'India', 'France', 'Japan', 'Canada', 'Germany', 'Italy', 'Spain', 'Australia']
        def cat_country(val):
            res = {f'is_country_{c.replace(" ", "_")}': 0 for c in top_countries}
            res.update({'is_other_country': 0, 'unknown_country': 0})
            val = str(val).strip()
            if val in ['Not Found', 'unknown', 'nan'] or len(val) > 80:
                res['unknown_country'] = 1
                return pd.Series(res)
            text = val.lower()
            for c in top_countries:
                if c.lower() in text:
                    res[f'is_country_{c.replace(" ", "_")}'] = 1
                    text = text.replace(c.lower(), '')
            if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['is_other_country'] = 1
            if sum(res.values()) == 0: res['unknown_country'] = 1
            return pd.Series(res)
        df_clean = pd.concat([df_clean, df_clean['Country'].apply(cat_country)], axis=1)

    # 5. ז'אנרים
    if 'genres' in df_clean.columns:
        top_genres = ['Drama', 'Comedy', 'Romance', 'Action', 'Documentary', 'Crime', 'Thriller', 'Horror', 'Adventure', 'Mystery']
        def cat_genre(val):
            res = {f'is_genre_{g}': 0 for g in top_genres}
            res.update({'is_other_genre': 0, 'unknown_genre': 0})
            val = str(val).strip()
            if val in ['Not Found', 'unknown', '\\N', 'nan'] or len(val) > 80:
                res['unknown_genre'] = 1
                return pd.Series(res)
            text = val.replace('[','').replace(']','').replace("'",'').replace('"','').lower()
            for g in top_genres:
                if g.lower() in text:
                    res[f'is_genre_{g}'] = 1
                    text = text.replace(g.lower(), '')
            if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['is_other_genre'] = 1
            if sum(res.values()) == 0: res['unknown_genre'] = 1
            return pd.Series(res)
        df_clean = pd.concat([df_clean, df_clean['genres'].apply(cat_genre)], axis=1)

    # 6. זמן ריצה ודירוג
    if 'averageRating' in df_clean.columns:
        df_clean['averageRating'] = pd.to_numeric(df_clean['averageRating'], errors='coerce')
    if 'runtimeMinutes' in df_clean.columns:
        df_clean['runtimeMinutes'] = pd.to_numeric(df_clean['runtimeMinutes'], errors='coerce')

    # 7. הסרת עמודות
    cols_to_drop = [
        'tconst', 'primaryTitle', 'plot', 'BoxOffice', 'numVotes',
        'budget', 'budget_parsed', 'startYear', 'Language', 'Country', 'genres'
    ]
    df_model = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

    return df_model


# --- קריאת הנתונים והפעלת הפונקציה ---
print("טוען את קובץ הנתונים הגולמי...")
raw_df = pd.read_csv('dataset.csv', low_memory=False)

print("מעביר את הנתונים דרך הפונקציה המעודכנת prepare_data()...")
processed_df = prepare_data(raw_df)
print(f"העיבוד הושלם! גודל הדאטה המעובד: {processed_df.shape}")

טוען את קובץ הנתונים הגולמי...
מעביר את הנתונים דרך הפונקציה המעודכנת prepare_data()...
העיבוד הושלם! גודל הדאטה המעובד: (133884, 54)


In [7]:
# ==========================================
# חלק 2: מחלקות חכמות, Pipeline ואימון עם דוח Folds מפורט
# ==========================================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# --- א. יצירת רכיבים מותאמים אישית (Custom Transformers) ---

class PolynomialFeaturesGenerator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_new = X.copy()
        if isinstance(X_new, pd.DataFrame) and 'runtimeMinutes' in X_new.columns:
            X_new['runtimeMinutes_sq'] = X_new['runtimeMinutes'] ** 2
        return X_new

class CastTargetEncoder(BaseEstimator, TransformerMixin):
    """
    מקודד שחקנים המשלב החלקה בייסיאנית והגנת קריסה (safe_parse) על NaN.
    """
    def __init__(self, min_count=5, smoothing_m=30):
        self.min_count = min_count
        self.smoothing_m = smoothing_m
        self.actor_means_ = {}
        self.global_mean_ = 0

    def fit(self, X, y):
        df_temp = pd.DataFrame({'actors': X['lead_actors_ids'], 'rating': y})
        
        # פונקציית Safe Parse למניעת קריסות
        def safe_parse(val):
            if pd.isna(val): 
                return []
            val_str = str(val).strip()
            if val_str in ['', 'nan', 'None']: 
                return []
            return [a.strip() for a in val_str.split(',') if a.strip()]
            
        df_temp['actors'] = df_temp['actors'].apply(safe_parse)
        df_exploded = df_temp.explode('actors').dropna(subset=['actors'])
        
        self.global_mean_ = y.mean()
        
        stats = df_exploded.groupby('actors')['rating'].agg(['mean', 'count'])
        stats = stats[stats['count'] >= self.min_count]
        
        stats['smoothed'] = ((stats['count'] * stats['mean']) + (self.smoothing_m * self.global_mean_)) / (stats['count'] + self.smoothing_m)
        self.actor_means_ = stats['smoothed'].to_dict()
        return self

    def transform(self, X):
        X_new = X.copy()
        
        def get_cast_mean(actors_str):
            if pd.isna(actors_str):
                return self.global_mean_
            val_str = str(actors_str).strip()
            if val_str in ['', 'nan', 'None']:
                return self.global_mean_
                
            actors = [a.strip() for a in val_str.split(',') if a.strip() != '']
            if not actors:
                return self.global_mean_
            
            actor_scores = [self.actor_means_.get(a, self.global_mean_) for a in actors]
            return np.mean(actor_scores)

        X_new['cast_avg_rating'] = X_new['lead_actors_ids'].apply(get_cast_mean)
        X_new = X_new.drop(columns=['lead_actors_ids']) 
        return X_new


# --- ב. הכנת הנתונים והפיצול ---
print("מכין נתונים ומפצל ל-Train ו-Test...")
processed_df_model = processed_df.dropna(subset=['averageRating']).copy()

y = processed_df_model['averageRating']
X = processed_df_model.drop(columns=['averageRating'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# --- ג. הרכבת ה-Pipeline ליער האקראי ---
print("\nמרכיב Pipeline למודל Random Forest...")
pipeline_rf = Pipeline([
    ('cast_encoder', CastTargetEncoder(min_count=5, smoothing_m=30)), 
    ('poly_features', PolynomialFeaturesGenerator()),
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('rf', RandomForestRegressor(random_state=42)) 
])


# --- ד. הגדרת מערכת המדידה הכפולה (Scoring) ---
# אנחנו דורשים מהמודל למדוד גם R2 וגם RMSE כדי שנוכל להדפיס את שניהם!
scoring_metrics = {
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

param_grid_rf = {
    'rf__n_estimators': [100],               
    'rf__max_depth': [15, 20, 25],     
    'rf__min_samples_split': [5, 10],  
    'rf__min_samples_leaf': [5, 10]    
}

print("\nמתחיל לאמן יער אקראי עם 10 Folds... (ממתין לסיום)")
grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=10, 
    scoring=scoring_metrics,
    refit='RMSE', # ההחלטה על המודל המנצח תתבצע לפי RMSE
    return_train_score=True, # הפקודה שמכריחה שמירת Train Scores!
    n_jobs=-1, 
    verbose=2 # יציג התקדמות בסיסית בזמן הריצה
)

grid_search_rf.fit(X_train, y_train)


# --- ה. הדפסת דוח Folds מפורט לבדיקת Overfitting ---
print("\n" + "="*60)
print(" דוח מעקב Overfitting לפי קיפולים (10 Folds) למודל המנצח")
print("="*60)

best_idx = grid_search_rf.best_index_
cv_res = grid_search_rf.cv_results_

for i in range(10):
    t_r2 = cv_res[f'split{i}_train_R2'][best_idx]
    v_r2 = cv_res[f'split{i}_test_R2'][best_idx]
    t_rmse = -cv_res[f'split{i}_train_RMSE'][best_idx]
    v_rmse = -cv_res[f'split{i}_test_RMSE'][best_idx]
    
    print(f"Fold {i+1:02d} | Train R²: {t_r2:.4f} vs Val R²: {v_r2:.4f}  ||  Train RMSE: {t_rmse:.4f} vs Val RMSE: {v_rmse:.4f}")

print("-" * 60)
mean_train_r2 = cv_res['mean_train_R2'][best_idx]
mean_val_r2 = cv_res['mean_test_R2'][best_idx]
print(f"ממוצע כללי | Train R²: {mean_train_r2:.4f} vs Val R²: {mean_val_r2:.4f}")
print("="*60)


# --- ו. הערכת ביצועים סופית ---
print("\n=== תוצאות האימון הסופיות (Random Forest) ===")
print(f"הפרמטרים הטובים ביותר: {grid_search_rf.best_params_}")

best_rf_model = grid_search_rf.best_estimator_
y_pred_test = best_rf_model.predict(X_test)

print("\n=== ביצועים על קבוצת הבדיקה (Test Set הנסתרת) ===")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred_test):.4f}")
print(f"R^2:  {r2_score(y_test, y_pred_test):.4f}")

מכין נתונים ומפצל ל-Train ו-Test...

מרכיב Pipeline למודל Random Forest...

מתחיל לאמן יער אקראי עם 10 Folds... (ממתין לסיום)
Fitting 10 folds for each of 12 candidates, totalling 120 fits

 דוח מעקב Overfitting לפי קיפולים (10 Folds) למודל המנצח
Fold 01 | Train R²: 0.3993 vs Val R²: 0.2670  ||  Train RMSE: 1.0040 vs Val RMSE: 1.0992
Fold 02 | Train R²: 0.3999 vs Val R²: 0.2711  ||  Train RMSE: 1.0015 vs Val RMSE: 1.1165
Fold 03 | Train R²: 0.3991 vs Val R²: 0.2657  ||  Train RMSE: 1.0033 vs Val RMSE: 1.1084
Fold 04 | Train R²: 0.3983 vs Val R²: 0.2872  ||  Train RMSE: 1.0039 vs Val RMSE: 1.0936
Fold 05 | Train R²: 0.4007 vs Val R²: 0.2568  ||  Train RMSE: 1.0015 vs Val RMSE: 1.1213
Fold 06 | Train R²: 0.3994 vs Val R²: 0.2713  ||  Train RMSE: 1.0023 vs Val RMSE: 1.1130
Fold 07 | Train R²: 0.3949 vs Val R²: 0.2833  ||  Train RMSE: 1.0077 vs Val RMSE: 1.0867
Fold 08 | Train R²: 0.3982 vs Val R²: 0.2698  ||  Train RMSE: 1.0038 vs Val RMSE: 1.1092
Fold 09 | Train R²: 0.3981 vs Val R²: 0.2

In [8]:
import pandas as pd
import os

# 1. שליפת הרכיבים המאומנים מתוך ה-Pipeline המנצח
best_pipeline = grid_search_rf.best_estimator_
trained_encoder = best_pipeline.named_steps['cast_encoder']
trained_poly = best_pipeline.named_steps['poly_features']
trained_imputer = best_pipeline.named_steps['imputer']
trained_scaler = best_pipeline.named_steps['scaler']

# 2. העברת נתוני האימון (X_train) בשרשרת העיבוד בדיוק כפי שהמודל חווה אותה
X_step1 = trained_encoder.transform(X_train)
X_step2 = trained_poly.transform(X_step1)
X_step3 = trained_imputer.transform(X_step2)
X_final_matrix = trained_scaler.transform(X_step3)

# 3. הרכבה חזרה לטבלה קריאה עם שמות העמודות הנכונים
final_feature_names = X_step2.columns 
df_excel = pd.DataFrame(X_final_matrix, columns=final_feature_names)

# 4. הוספת עמודת המטרה (הציון ב-IMDB) בסוף כדי שתראה מה המודל לומד לחזות
df_excel['TARGET_averageRating'] = y_train.values

# 5. שמירת הקובץ (תוכל למצוא אותו בתיקייה של הג'ופיטר שלך)
excel_filename = 'model_final_training_data.csv'
df_excel.to_csv(excel_filename, index=False)

print(f"הקובץ נוצר בהצלחה!")
print(f"חפש בתיקייה שלך את הקובץ: {excel_filename} ופתח אותו באקסל.")
print(f"מידות הטבלה שהמודל למד ממנה: {df_excel.shape[0]} סרטים, {df_excel.shape[1]} עמודות.")

הקובץ נוצר בהצלחה!
חפש בתיקייה שלך את הקובץ: model_final_training_data.csv ופתח אותו באקסל.
מידות הטבלה שהמודל למד ממנה: 92448 סרטים, 55 עמודות.
